# Suavização, Remoção de Ruído e Detecção de Bordas

**Disciplina:** Visão Computacional  
**Programa:** Residência em TIC43 (iRede) — **Unidade:** 2 | **Capítulo:** 1 | **Tarefa:** 1  
**Aluno:** Marcio Barbosa

---
## Parte 1 — Seleção das Imagens

### Como as imagens foram escolhidas

A seleção aconteceu em duas etapas.

Na primeira, rodei um algoritmo automático sobre as 120 imagens do dataset, usando a **variância do Laplaciano** para estimar o nível de ruído de cada imagem e o **desvio padrão dos tons de cinza** como indicador de contraste. As 20 com maiores índices de degradação foram listadas para análise.

Na segunda etapa, fiz uma curadoria visual dessas 20 candidatas. Percebi que várias delas tinham pontuação alta por causa do **contraste artificial das sombras** projetadas na mesa — e não por ruído real de sensor. Por isso, escolhi a combinação final pensando em cobrir tipos de degradação distintos, tornando a prática mais rica.

O código de ranqueamento está na célula abaixo.

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd

img_folder = '../data/novas/'
arquivos = [f for f in os.listdir(img_folder) if f.endswith('.jpg')]

analise_dados = []

print("Iniciando análise técnica do dataset...")

for arq in arquivos:
    path = os.path.join(img_folder, arq)
    img = cv2.imread(path)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    brilho_medio = np.mean(gray)
    contraste = np.std(gray)

    # Estimador de ruído: variância do Laplaciano
    # Quanto maior, mais variação de alta frequência (ruído) a imagem tem
    ruido = cv2.Laplacian(gray, cv2.CV_64F).var()

    # Densidade de bordas via Canny (pode indicar falsos positivos por ruído)
    bordas = cv2.Canny(gray, 50, 150)
    densidade_borda = (np.sum(bordas == 255) / bordas.size) * 100

    analise_dados.append({
        'Arquivo': arq,
        'ID': arq.split('_')[1][:6],
        'Ruido_Laplaciano': round(ruido, 2),
        'Densidade_Borda': round(densidade_borda, 2),
        'Contraste': round(contraste, 2),
        'Brilho': round(brilho_medio, 2)
    })

df = pd.DataFrame(analise_dados)
df_piores_candidatas = df.sort_values(by=['Ruido_Laplaciano', 'Contraste'], ascending=[False, True])

print("\n--- TOP 20 IMAGENS MAIS DEGRADADAS ---")
print(df_piores_candidatas[['ID', 'Ruido_Laplaciano', 'Densidade_Borda', 'Contraste', 'Brilho']].head(20))

In [ ]:
import cv2
import numpy as np
import os
import pandas as pd

img_folder = '../data/novas/'
arquivos = [f for f in os.listdir(img_folder) if f.endswith('.jpg')]

analise_dados = []

print("Iniciando análise técnica do dataset...")

for arq in arquivos:
    path = os.path.join(img_folder, arq)
    img = cv2.imread(path)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    brilho_medio = np.mean(gray)
    contraste = np.std(gray)

    # Estimador de ruído: variância do Laplaciano
    # Quanto maior, mais variação de alta frequência (ruído) a imagem tem
    ruido = cv2.Laplacian(gray, cv2.CV_64F).var()

    # Densidade de bordas via Canny (pode indicar falsos positivos por ruído)
    bordas = cv2.Canny(gray, 50, 150)
    densidade_borda = (np.sum(bordas == 255) / bordas.size) * 100

    analise_dados.append({
        'Arquivo': arq,
        'ID': arq.split('_')[1][:6],
        'Ruido_Laplaciano': round(ruido, 2),
        'Densidade_Borda': round(densidade_borda, 2),
        'Contraste': round(contraste, 2),
        'Brilho': round(brilho_medio, 2)
    })

df = pd.DataFrame(analise_dados)
df_piores_candidatas = df.sort_values(by=['Ruido_Laplaciano', 'Contraste'], ascending=[False, True])

print("\n--- TOP 20 IMAGENS MAIS DEGRADADAS ---")
print(df_piores_candidatas[['ID', 'Ruido_Laplaciano', 'Densidade_Borda', 'Contraste', 'Brilho']].head(20))

### As 4 imagens selecionadas

Depois da análise automática e da curadoria visual, estas foram as imagens escolhidas:

---

**Imagem 1 — 20260507_084538.jpg**  
- **Classe:** A — Adenium (flores grandes, tronco espesso)  
- **Degradação observada:** Granulação fina sobre a textura do tronco e variações de intensidade nas sombras projetadas na mesa.  
- **Hipótese:** O processo de demosaicing e a compressão JPEG criaram artefatos de alta frequência que se misturam com a textura rugosa da planta, dificultando a separação entre ruído e detalhe real.

---

**Imagem 2 — 20260507_084114.jpg**  
- **Classe:** A — Folhagem densa (planta arbustiva de pequenas folhas)  
- **Degradação observada:** Ruído de quantização visível nas regiões escuras do fundo, onde a relação sinal-ruído do sensor é mais baixa.  
- **Hipótese:** O ganho eletrônico (ISO) elevado em ambiente de pouca luz, combinado com erros do conversor A/D, gerou pixels com intensidades aleatórias que os detectores de borda interpretam como gradientes estruturais.

---

**Imagem 3 — 20260507_085738.jpg**  
- **Classe:** B — Orquídea (haste fina, flores pequenas)  
- **Condição de captura:** Ambiente coberto, iluminação natural difusa  
- **Degradação observada:** Ruído granular distribuído uniformemente pelo fundo bege, característico de captura com ISO elevado em baixa luminosidade.  
- **Hipótese:** Ruído térmico do sensor CMOS (dark current), que aparece como variações aleatórias nas regiões de cor uniforme. É o cenário ideal para validar os filtros de média e gaussiano.

---

**Imagem 4 — 20260507_084445.jpg**  
- **Classe:** B — Orquídea (haste fina, flores pequenas)  
- **Condição de captura:** Sol direto, iluminação intensa  
- **Degradação observada:** Sombras duras de alto contraste sobrepostas ao ruído eletrônico de base, gerando alta densidade de falsos positivos na detecção de bordas.  
- **Hipótese:** A degradação é mista — ruído eletrônico base somado a gradientes falsos criados pelas sombras. Esses gradientes enganam os operadores de Sobel e Prewitt, simulando bordas que não pertencem à planta.

---

### Por que essa combinação faz sentido

As quatro imagens cobrem três tipos de desafio diferentes:

- **Classe A** traz estruturas de borda grossa e curva (tronco do Adenium) e textura densa em fundo escuro — bom para observar o comportamento dos filtros em regiões de alto contraste real.
- **Classe B** aparece duas vezes com condições de luz opostas — ruído puro de sensor (085738) versus ruído misturado com sombras duras (084445) — o que permite comparar como os mesmos filtros se comportam sob tipos de degradação diferentes na mesma classe botânica.

Essa variação vai tornar a análise das Partes 3 e 4 mais interessante, já que cada operador de borda vai reagir de forma diferente em cada cenário.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

selected_files = [
    '20260507_084538.jpg',
    '20260507_084114.jpg',
    '20260507_085738.jpg',
    '20260507_084445.jpg'
]

img_folder = '../data/novas/'

fig, axes = plt.subplots(2, 2, figsize=(20, 15))
axes = axes.ravel()
fig.suptitle('Parte 1 — Imagens Selecionadas', fontsize=18, fontweight='bold')

for i, arq in enumerate(selected_files):
    path = os.path.join(img_folder, arq)
    img = cv2.imread(path)
    if img is not None:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img_rgb)
        axes[i].set_title(arq.split('_')[1][:6], fontsize=14)
    else:
        axes[i].set_title("ERRO: Não encontrada")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

---
## Parte 2 — Aplicação dos Filtros de Suavização

Para cada imagem, vou aplicar os seis filtros exigidos e registrar a variância dos níveis de cinza antes e depois de cada um. Os resultados visuais e a tabela estão logo abaixo.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
from IPython.display import display

selected_files = [
    '20260507_084538.jpg',
    '20260507_084114.jpg',
    '20260507_085738.jpg',
    '20260507_084445.jpg'
]
img_folder = '../data/novas/'

resultados_var = []

for arq in selected_files:
    path = os.path.join(img_folder, arq)
    img = cv2.imread(path)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Filtro da Média — calcula a média aritmética dos pixels vizinhos
    # Kernel maior = mais suavização, mas borra mais as bordas
    media_3 = cv2.blur(gray, (3, 3))
    media_5 = cv2.blur(gray, (5, 5))

    # Filtro Gaussiano — média ponderada, pixels do centro têm mais peso
    # Sigma maior = curva mais larga = mais suavização
    # (0,0) faz o OpenCV calcular o tamanho ideal do kernel com base no sigma
    gauss_1 = cv2.GaussianBlur(gray, (0, 0), sigmaX=1)
    gauss_3 = cv2.GaussianBlur(gray, (0, 0), sigmaX=3)

    # Filtro da Mediana — substitui cada pixel pelo valor do meio após ordenar a vizinhança
    # Não-linear: muito eficaz contra ruído impulsivo sem borrar tanto as bordas
    mediana_3 = cv2.medianBlur(gray, 3)
    mediana_5 = cv2.medianBlur(gray, 5)

    resultados_var.append({
        'ID': arq.split('_')[1][:6],
        'Original': round(np.var(gray), 2),
        'Média 3x3': round(np.var(media_3), 2),
        'Média 5x5': round(np.var(media_5), 2),
        'Gauss σ=1': round(np.var(gauss_1), 2),
        'Gauss σ=3': round(np.var(gauss_3), 2),
        'Mediana 3x3': round(np.var(mediana_3), 2),
        'Mediana 5x5': round(np.var(mediana_5), 2)
    })

    titulos = ['Original', 'Média 3x3', 'Média 5x5', 'Gauss σ=1', 'Gauss σ=3', 'Mediana 3x3', 'Mediana 5x5']
    imagens = [gray, media_3, media_5, gauss_1, gauss_3, mediana_3, mediana_5]

    fig, axes = plt.subplots(1, 7, figsize=(28, 4))
    fig.suptitle(f'Filtros Espaciais — {arq.split("_")[1][:6]}', fontsize=14, fontweight='bold')

    for i in range(7):
        axes[i].imshow(imagens[i], cmap='gray')
        axes[i].set_title(titulos[i], fontsize=11)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

df_variancia = pd.DataFrame(resultados_var)
print("\n--- VARIÂNCIA DOS NÍVEIS DE CINZA ---")
display(df_variancia)

### O que os resultados mostraram

**Qual filtro reduziu mais o ruído?**  
Os filtros Média 5x5 e Gaussiano σ=3 foram os mais agressivos na redução de granulação geral. Isso faz sentido: quanto maior a área de vizinhança considerada, mais o algoritmo dilui os picos de alta frequência. O efeito ficou mais evidente na imagem 085738 (Orquídea em ambiente coberto), onde o fundo bege ruidoso ficou nitidamente mais limpo.  
Para ruídos mais isolados e pontuais — como o ruído de quantização no fundo escuro da 084114 — o Filtro da Mediana 5x5 foi o mais eficaz. Por ser não-linear, ele elimina o pixel anômalo sem "espalhar" o problema para os vizinhos.

**Qual preservou melhor as bordas?**  
O Filtro da Mediana 3x3 foi o melhor nesse quesito. A lógica por trás disso é que ele não faz nenhuma operação aritmética — apenas ordena os valores e escolhe o do meio. Com isso, as transições abruptas de intensidade (que definem as bordas) tendem a ser preservadas.  
O Gaussiano σ=1 ficou em segundo lugar, borrando um pouco menos que a Média porque seu peso é distribuído de forma decrescente a partir do centro do kernel.

**O que aconteceu com a variância?**  
Como mostra a tabela acima, a variância caiu em todos os casos após a filtragem. Isso era esperado: os filtros de suavização atuam como filtros passa-baixa, cortando as variações abruptas de intensidade. Ao fazer isso, os tons da imagem convergem para valores médios locais, tornando a distribuição de intensidades mais homogênea — e, por consequência, a variância menor.  
A regra foi consistente: filtros mais agressivos (Média 5x5, Gaussiano σ=3) produziram as maiores quedas de variância.

---
## Parte 3 — Detecção de Bordas sem Suavização Prévia

Aqui aplico os três operadores diretamente na imagem original, sem nenhum pré-processamento. O objetivo é observar como cada um se comporta na presença de ruído real.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

selected_files = [
    '20260507_084538.jpg',
    '20260507_084114.jpg',
    '20260507_085738.jpg',
    '20260507_084445.jpg'
]
img_folder = '../data/novas/'

# Kernels do Prewitt (não disponível nativamente no OpenCV)
kernelx_prewitt = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=np.float32)
kernely_prewitt = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], dtype=np.float32)

print("Parte 3 — Detecção de bordas sem suavização prévia...")

for arq in selected_files:
    path = os.path.join(img_folder, arq)
    img = cv2.imread(path)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Sobel: detecta gradiente em X (bordas verticais) e Y (bordas horizontais)
    # O peso central 2 no kernel introduz uma leve suavização implícita
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobelx, sobely)

    # Prewitt: similar ao Sobel, mas sem ponderação central
    # Mais sensível ao ruído por tratar todos os vizinhos com o mesmo peso
    prewittx = cv2.filter2D(gray, cv2.CV_64F, kernelx_prewitt)
    prewitty = cv2.filter2D(gray, cv2.CV_64F, kernely_prewitt)
    prewitt_mag = cv2.magnitude(prewittx, prewitty)

    # Canny: aplica supressão de não-máximos e histerese — gera bordas mais limpas
    # Testando dois limiares para observar o trade-off sensibilidade x ruído
    canny_baixo = cv2.Canny(gray, 50, 150)   # mais sensível
    canny_alto  = cv2.Canny(gray, 100, 200)  # mais restritivo

    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    fig.suptitle(f'Parte 3: Bordas Sem Suavização — {arq.split("_")[1][:6]}', fontsize=14, fontweight='bold')

    axes[0, 0].imshow(np.abs(sobelx), cmap='gray');    axes[0, 0].set_title('Sobel X')
    axes[0, 1].imshow(np.abs(sobely), cmap='gray');    axes[0, 1].set_title('Sobel Y')
    axes[0, 2].imshow(sobel_mag, cmap='gray');         axes[0, 2].set_title('Sobel — Magnitude')

    axes[1, 0].imshow(np.abs(prewittx), cmap='gray');  axes[1, 0].set_title('Prewitt X')
    axes[1, 1].imshow(np.abs(prewitty), cmap='gray');  axes[1, 1].set_title('Prewitt Y')
    axes[1, 2].imshow(prewitt_mag, cmap='gray');       axes[1, 2].set_title('Prewitt — Magnitude')

    axes[2, 0].imshow(canny_baixo, cmap='gray');       axes[2, 0].set_title('Canny (50, 150)')
    axes[2, 1].imshow(canny_alto, cmap='gray');        axes[2, 1].set_title('Canny (100, 200)')
    axes[2, 2].imshow(gray, cmap='gray');              axes[2, 2].set_title('Original')

    for ax in axes.flat:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

### O que foi observado

**Sensibilidade ao ruído — Sobel vs Prewitt**  
O Prewitt se mostrou mais ruidoso nas três situações. Como ele trata todos os pixels da vizinhança com o mesmo peso, qualquer variação de intensidade — mesmo que seja ruído — é tratada como gradiente válido. O Sobel tem uma vantagem aqui: o peso 2 no centro do kernel funciona como uma suavização implícita, o que estabiliza um pouco a resposta em regiões de textura complexa.

**Bordas falsas**  
As imagens 085738 e 084114 foram as mais problemáticas. Sem filtragem prévia, tanto o Sobel quanto o Prewitt detectaram o ruído do fundo como se fossem bordas reais, misturando o objeto de interesse com o ruído de fundo. O Canny se saiu melhor nesse quesito por conta da etapa de supressão de não-máximos, que elimina respostas fracas antes de tomar a decisão final.

**Comparação entre os limiares do Canny**  
Com limiar baixo (50, 150), o detector capturou quase todo o ruído granular, tornando difícil distinguir o objeto do fundo. Com limiar alto (100, 200), o ruído foi reduzido, mas ao custo de fragmentar as bordas reais da planta — as hastes finas da orquídea e os contornos do tronco do Adenium ficaram com lacunas. Nenhum dos dois parâmetros é ideal sem suavização prévia, o que motiva a Parte 4.

---
## Parte 4 — Detecção de Bordas após Suavização

Com base nos resultados da Parte 2, o **Filtro da Mediana 5x5** foi escolhido como etapa de pré-processamento. A justificativa é que ele foi o mais eficaz na remoção de ruído impulsivo e pontual — que é o tipo predominante nessas imagens — sem destruir as bordas estruturais da planta.

A seguir, o pipeline completo: imagem original → suavização → detecção com Sobel, Prewitt e Canny.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from IPython.display import display

selected_files = [
    '20260507_084538.jpg',
    '20260507_084114.jpg',
    '20260507_085738.jpg',
    '20260507_084445.jpg'
]
img_folder = '../data/novas/'

kernelx_prewitt = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=np.float32)
kernely_prewitt = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], dtype=np.float32)

fig, axes = plt.subplots(4, 5, figsize=(24, 20))
col_titles = ['Original', 'Mediana 5x5', 'Sobel', 'Prewitt', 'Canny']

estatisticas_bordas = []

for i, arq in enumerate(selected_files):
    path = os.path.join(img_folder, arq)
    img = cv2.imread(path)
    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Filtro escolhido: Mediana 5x5
    suavizada = cv2.medianBlur(gray, 5)

    # Sobel pós-filtro
    sobelx = cv2.Sobel(suavizada, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(suavizada, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = cv2.magnitude(sobelx, sobely)

    # Prewitt pós-filtro
    prewittx = cv2.filter2D(suavizada, cv2.CV_64F, kernelx_prewitt)
    prewitty = cv2.filter2D(suavizada, cv2.CV_64F, kernely_prewitt)
    prewitt_mag = cv2.magnitude(prewittx, prewitty)

    # Canny pós-filtro (mesmo limiar da Parte 3 para comparação direta)
    canny_bruto = cv2.Canny(gray, 50, 150)
    canny_limpo = cv2.Canny(suavizada, 50, 150)

    # Estatística: contagem de pixels de borda antes e depois
    qtd_bruto = np.sum(canny_bruto == 255)
    qtd_limpo = np.sum(canny_limpo == 255)
    reducao = (1 - qtd_limpo / qtd_bruto) * 100 if qtd_bruto > 0 else 0

    estatisticas_bordas.append({
        'ID': arq.split('_')[1][:6],
        'Bordas sem filtro': qtd_bruto,
        'Bordas com Mediana 5x5': qtd_limpo,
        'Redução (%)': f"{reducao:.1f}%"
    })

    imagens_plot = [gray, suavizada, sobel_mag, prewitt_mag, canny_limpo]
    for j in range(5):
        axes[i, j].imshow(imagens_plot[j], cmap='gray')
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=14, fontweight='bold')

    axes[i, 0].text(-0.1, 0.5, arq.split('_')[1][:6],
                    transform=axes[i, 0].transAxes,
                    fontsize=13, fontweight='bold',
                    va='center', ha='right', rotation=90)

plt.tight_layout()
fig.suptitle('Parte 4 — Detecção de Bordas Pós-Suavização', fontsize=18, fontweight='bold', y=1.01)
plt.show()

df_estatistica = pd.DataFrame(estatisticas_bordas)
print("\n--- REDUÇÃO DE BORDAS FALSAS (CANNY) ---")
display(df_estatistica)

### Comparação e síntese final

**A suavização melhorou a detecção?**  
Sim, de forma bastante clara. A tabela acima mostra a redução percentual de pixels detectados como borda após aplicar a Mediana 5x5. Essa queda não significa perda de informação — significa que os falsos positivos gerados pelo ruído foram removidos. As bordas que restaram são mais contínuas e mais fáceis de interpretar.

**Qual operador foi mais robusto ao ruído?**  
O Sobel. Mesmo sem suavização, ele apresentou resultados mais estáveis que o Prewitt graças à sua ponderação central. Após a filtragem, a diferença entre os dois diminuiu, mas o Sobel ainda manteve gradientes mais limpos nas regiões de textura densa (Lavanda/084114).

**Qual apresentou melhor definição estrutural?**  
O Canny, quando combinado com a suavização prévia. A etapa de supressão de não-máximos do algoritmo gera bordas com espessura de 1 pixel, muito mais precisas que os gradientes espessos do Sobel e Prewitt. Para uma tarefa de segmentação ou extração de características, o Canny pós-filtro seria a escolha mais indicada.

---

### Conclusão

Este pipeline mostrou na prática algo que fica mais claro quando aplicado em imagens reais: a qualidade da detecção de bordas depende diretamente da qualidade da imagem de entrada. Aplicar um operador de gradiente diretamente em uma imagem ruidosa é como tentar ouvir uma conversa com muito barulho de fundo — você capta tudo, mas não consegue separar o que importa.

A suavização, especialmente com a Mediana, funcionou como um filtro que reduz o ruído de fundo sem apagar os detalhes estruturais da planta. Com isso, os operadores conseguiram focar nas transições de intensidade que realmente correspondem a bordas físicas do objeto.

Para as próximas etapas do curso — segmentação e classificação — esse pré-processamento vai ser essencial, principalmente em datasets capturados em condições variadas de iluminação como este.